# agentic_alpha_intraday_execution_label_v1

Hypothesis: the prior 15m model used a noisy next-bar label. This experiment upgrades the intraday execution label to triple-barrier + execution-improvement and adds same-day context features while keeping qlib as the data and recorder system of record.

The workflow uses qlib.init, qlib.data.D.features, DatasetH/DataHandlerLP/QlibDataLoader audit, LGBMClassifier, XGBClassifier, and R.start. Large predictions stay in the qlib recorder.

In [1]:
from pathlib import Path
import sys

import qlib
from qlib.workflow import R

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.workflow.intraday_execution_label_model import (
    EXPERIMENT,
    IntradayExecutionLabelModelSpec,
    run_intraday_execution_label_model_card,
)

spec = IntradayExecutionLabelModelSpec(
    experiment=EXPERIMENT,
    output_dir=ROOT / "research" / EXPERIMENT,
    recorder_uri=ROOT / "research" / EXPERIMENT / "qlib_recorder",
    provider_uri=ROOT / "data" / "qlib" / "soxl_intraday_15min",
)
# qlib.init and R.start are called inside the workflow; triple_barrier labels are generated from qlib 15m bars.
summary = run_intraday_execution_label_model_card(spec)
summary["recorder_id"]

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[75100:MainThread](2026-07-04 14:37:04,696) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[75100:MainThread](2026-07-04 14:37:04,699) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[75100:MainThread](2026-07-04 14:37:04,699) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


[75100:MainThread](2026-07-04 14:37:04,793) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[75100:MainThread](2026-07-04 14:37:04,956) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl_intraday_15min')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[75100:MainThread](2026-07-04 14:38:10,727) INFO - qlib.workflow - [exp.py:258] - Experiment 502101112689773134 starts running ...


[75100:MainThread](2026-07-04 14:38:10,803) INFO - qlib.workflow - [recorder.py:345] - Recorder 13e134d6818f4dc39622b1c3458b979e starts running under Experiment 502101112689773134 ...


[75100:MainThread](2026-07-04 14:38:10,888) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


'13e134d6818f4dc39622b1c3458b979e'

In [2]:
import pandas as pd

card = ROOT / "research" / EXPERIMENT
display(pd.read_csv(card / summary["sidecars"]["model_selection"]))
display(pd.read_csv(card / summary["sidecars"]["backtest_walk_forward_metrics"]))
print((card / "conclusion.md").read_text(encoding="utf-8"))

,model,threshold,valid_ic,test_ic,valid_trades,valid_trade_days,valid_coverage_days,valid_net_return_pct,valid_gross_return_pct,valid_max_drawdown_pct,...,test_max_drawdown_pct,test_daily_sharpe,test_win_rate,test_avg_trade_bps,test_worst_trade_bps,test_avg_execution_improvement_bps,test_profit_outcome_ratio,test_stop_outcome_ratio,selection_score,selection_eligible
0,lgb,0.60000,-0.021453,-0.005308,84,26,250,-21.244815,-6.806557,-26.135424,...,-11.204859,2.958629,0.600000,18.612276,-408.006288,-41.881350,0.442857,0.428571,-1042.965192,False
1,lgb_xgb_consensus,0.55000,-0.021470,-0.005459,108,31,250,-42.710942,-28.832082,-44.495970,...,-13.298019,0.762550,0.553191,4.862468,-408.006288,-28.628222,0.404255,0.414894,-1082.718309,False
2,xgb,0.48882,-0.020062,-0.008311,622,132,250,-85.675116,-50.159193,-86.896430,...,-67.983760,-1.682088,0.467601,-10.981079,-944.406761,15.817367,0.336252,0.448336,-1140.031080,False


,fold,test_year,train_start,train_end,valid_start,valid_end,test_start,test_end,selected_model,threshold,selection_uses_test_segment,valid_net_return_pct,valid_max_drawdown_pct,test_net_return_pct,test_max_drawdown_pct,test_avg_execution_improvement_bps,test_10bp_net_return_pct,test_20bp_net_return_pct,test_50bp_net_return_pct
0,wf_2023,2023,2019-01-02,2021-12-31,2022-01-01,2022-12-31,2023-01-01,2023-12-31,xgb,0.60,False,7.329451,-15.759134,1.670717,-9.000925,50.600417,3.411506,1.670717,-3.387646
1,wf_2024,2024,2019-01-02,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,lgb_xgb_consensus,0.55,False,13.381048,-18.392286,-26.228867,-42.797896,38.387574,-6.198548,-26.228867,-64.166517
2,wf_2025,2025,2019-01-02,2023-12-31,2024-01-01,2024-12-31,2025-01-01,2025-12-31,lgb,0.60,False,14.201253,-11.085522,-48.812771,-50.930686,8.495429,-41.319426,-48.812771,-66.052729
3,wf_2026,2026,2019-01-02,2024-12-31,2025-01-01,2025-12-31,2026-01-01,2026-07-02,lgb,0.60,False,-21.244815,-26.135424,12.909799,-11.204859,-41.881350,21.078828,12.909799,-8.474985


# 中文结论

- 做了什么：把 15m 日内模型的 label 从下一根 bar 涨跌升级为 triple-barrier + execution-improvement，并加入 open gap、VWAP 距离、realized vol 扩张、前 30/60 分钟趋势质量、量价吸收/派发、SOXX lead-lag 等同日上下文因子。
- 核心指标：validation 选择 lgb，valid return_pct=-21.2448、test return_pct=12.9098、test max_drawdown_pct=-11.2049。
- walk-forward：按 train 到 T-2、valid T-1、test T 记录 2023-2026，正收益比例=0.50。
- 是否晋升：不晋升；decision=not_promoted_walk_forward_not_proven。它是执行层候选，不替代日级 open-to-open 仓位模型。
- 风险：当前分钟级外部上下文只有 SOXL/SOXX，QQQ/SMH 与 NVDA/AVGO/AMD/TSM/ASML 还没有分钟级 qlib store；真实执行还缺 bid/ask replay、排队成交、延迟和极端波动成交验证。
- qlib recorder：13e134d6818f4dc39622b1c3458b979e

